In [1]:
!unzip /content/reward.zip

Archive:  /content/reward.zip
   creating: reward/
  inflating: reward/test.json        
  inflating: reward/train.json       
  inflating: reward/valid.json       


In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [3]:
import torch
import os

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
from datasets import load_dataset

data = load_dataset('json', data_files='/content/reward/train.json',split='train')
print(data)

def f(data):
    #区分两种生成结果
    chosen = data['question'] + data['response_chosen'].swapcase()
    rejected = data['question'] + data['response_rejected']

    chosen_input = tokenizer(chosen,
                         max_length=300,
                         padding='max_length',
                         truncation=True,
                         return_tensors='pt')
    rejected_input = tokenizer(rejected,
                         max_length=300,
                         padding='max_length',
                         truncation=True,
                         return_tensors='pt')

    # print(chosen_input['attention_mask'])

    return {
        'chosen_input_ids': chosen_input['input_ids'],
        'chosen_attention_mask': chosen_input['attention_mask'],
        'rejected_input_ids': rejected_input['input_ids'],
        'rejected_attention_mask': rejected_input['attention_mask']
    }


dataset = data.map(f)
dataset.set_format('torch')

print(dataset)


def f(data):
  chosen_input_ids = [i['chosen_input_ids'] for i in data]
  chosen_attention_mask = [i['chosen_attention_mask'] for i in data]
  rejected_input_ids = [i['rejected_input_ids'] for i in data]
  rejected_attention_mask = [i['rejected_attention_mask'] for i in data]

  input_ids = torch.stack(chosen_input_ids + rejected_input_ids, dim=0)
  attention_mask = torch.stack(chosen_attention_mask +
                                rejected_attention_mask,
                                dim=0)


  return {'input_ids': input_ids, 'attention_mask': attention_mask}


loader = torch.utils.data.DataLoader(dataset,
                                     collate_fn=f,
                                     batch_size=4,
                                     shuffle=True,
                                     drop_last=True)

len(loader)
next(iter(loader))

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['question', 'response_chosen', 'response_rejected'],
    num_rows: 3800
})


Map:   0%|          | 0/3800 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'response_chosen', 'response_rejected', 'chosen_input_ids', 'chosen_attention_mask', 'rejected_input_ids', 'rejected_attention_mask'],
    num_rows: 3800
})


{'input_ids': tensor([[[20998,   246, 28156,  ...,   240,   111,   161]],
 
         [[22887,   120,   163,  ..., 50256, 50256, 50256]],
 
         [[20998,   111, 33768,  ..., 50256, 50256, 50256]],
 
         ...,
 
         [[22887,   120,   163,  ..., 50256, 50256, 50256]],
 
         [[20998,   111, 33768,  ..., 50256, 50256, 50256]],
 
         [[  162,   224,    96,  ..., 29785,   112,   164]]]),
 'attention_mask': tensor([[[1, 1, 1,  ..., 1, 1, 1]],
 
         [[1, 1, 1,  ..., 0, 0, 0]],
 
         [[1, 1, 1,  ..., 0, 0, 0]],
 
         ...,
 
         [[1, 1, 1,  ..., 0, 0, 0]],
 
         [[1, 1, 1,  ..., 0, 0, 0]],
 
         [[1, 1, 1,  ..., 1, 1, 1]]])}

In [5]:
class CriticModel(torch.nn.Module):

    def __init__(self):
        super().__init__()

        from transformers import AutoModel
        self.rwtransformer = AutoModel.from_pretrained('facebook/opt-350m',
                                                       dropout=0.0)

        self.v_head = torch.nn.Linear(512, 1, bias=False)

    def forward(self, input_ids, attention_mask):

        input_ids = input_ids.squeeze(1)
        attention_mask = attention_mask.squeeze(1)

        value = self.rwtransformer(
            input_ids=input_ids,
            attention_mask=attention_mask).last_hidden_state
        value = self.v_head(value).squeeze(-1)

        loss_sum = 0.0
        value_chosen_sum = 0.0
        value_rejected_sum = 0.0
        for input_ids_chosen, input_ids_rejected, value_chosen, value_rejected in zip(
                input_ids[:4], input_ids[4:], value[:4], value[4:]):

            # 找出每条回答中的起止索引
            try:
                start = (
                    input_ids_chosen == input_ids_rejected).tolist().index(False)
            except ValueError:
                start = 0

            try:
                end_chosen = input_ids_chosen.tolist().index(
                    tokenizer.eos_token_id) + 1
            except ValueError:
                end_chosen = len(input_ids_chosen)

            try:
                end_rejected = input_ids_rejected.tolist().index(
                    tokenizer.eos_token_id) + 1
            except ValueError:
                end_rejected = len(input_ids_rejected)


            end = max(end_chosen, end_rejected)

            value_chosen = value_chosen[start:end]
            value_rejected = value_rejected[start:end]

            loss = value_chosen - value_rejected
            loss = -torch.nn.functional.logsigmoid(loss).mean()

            loss_sum += loss
            value_chosen_sum += value_chosen.mean().item()
            value_rejected_sum += value_rejected.mean().item()

        return loss_sum / 4, value_chosen_sum, value_rejected_sum


model_critic = CriticModel()


config.json:   0%|          | 0.00/371 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

In [6]:
from transformers import get_scheduler
from accelerate import Accelerator


def f():
    params_decay = []
    params = []
    for name, param in model_critic.named_parameters():
        if 'bias' in name or 'norm.weight' in name:
            params.append(param)
            continue
        params_decay.append(param)

    return [{
        'params': params_decay,
        'weight_decay': 0.1
    }, {
        'params': params,
        'weight_decay': 0.0
    }]


optimizer = torch.optim.Adam(f(), lr=5e-5, betas=(0.9, 0.95))

scheduler = get_scheduler(name='cosine',
                          optimizer=optimizer,
                          num_warmup_steps=0,
                          num_training_steps=500)

accelerator = Accelerator(gradient_accumulation_steps=16,
                          mixed_precision='fp16')

model_critic, loader, optimizer, scheduler = accelerator.prepare(
    model_critic, loader, optimizer, scheduler)

model_critic.train()

CriticModel(
  (rwtransformer): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_fea

In [8]:
for i, data in enumerate(loader):
    with accelerator.accumulate(model_critic):
        # print(data)
        loss, value_chosen_sum, value_rejected_sum = model_critic(**data)
        accelerator.backward(loss)

        if accelerator.sync_gradients:
            accelerator.clip_grad_norm_(model_critic.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    if (i + 1) % 100 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(i, len(loader), loss.item(), lr, value_chosen_sum,
              value_rejected_sum)

    if i == 2000:
        break

# torch.save(model_critic.to('cpu'), 'model/critic')
model_actor.save_pretrained('model/critic')

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)